In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Load and preprocess the data
mitbih_train = pd.read_csv('mitbih_train.csv', header=None)
mitbih_test = pd.read_csv('mitbih_test.csv', header=None)

# Separate features and labels
X_train = mitbih_train.iloc[:, :-1].values
y_train = mitbih_train.iloc[:, -1].values

X_test = mitbih_test.iloc[:, :-1].values
y_test = mitbih_test.iloc[:, -1].values

# Initial train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

# Apply SMOTE to balance the classes in the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# Standardize data
scaler = StandardScaler()
X_train_smote = scaler.fit_transform(X_train_smote)
X_test = scaler.transform(X_test)


In [2]:
# Reshape the original data for Conv2D
X_train_reshaped = X_train_smote.reshape(-1, 17, 11, 1)
X_test_reshaped = X_test.reshape(-1, 17, 11, 1)


In [3]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input, Multiply, GlobalAveragePooling2D

# Attention mechanism
def attention_layer(inputs):
    attention_probs = Dense(units=inputs.shape[-1], activation='softmax')(inputs)
    attention_mul = Multiply()([inputs, attention_probs])
    return attention_mul

# CNN-Attention Model
def create_cnn_attention_model():
    inputs = Input(shape=(17, 11, 1))
    
    # Convolutional layers
    x = Conv2D(32, kernel_size=(3, 3), activation='relu')(inputs)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    x = Dropout(0.25)(x)
    
    x = Conv2D(64, kernel_size=(2, 2), activation='relu')(x)
    x = MaxPooling2D(pool_size=(2, 2))(x)
    x = Dropout(0.25)(x)
    
    # Flatten the convolutional output
    x = Flatten()(x)
    
    # Apply attention mechanism
    attention = attention_layer(x)
    
    # Dense layers
    x = Dense(128, activation='relu')(attention)
    x = Dropout(0.5)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Output layer
    outputs = Dense(5, activation='softmax')(x)  # 5 classes for MIT-BIH
    
    # Build the model
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_attention_model = create_cnn_attention_model()
cnn_attention_model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 17, 11, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 15, 9, 32) │        320 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 7, 4, 32)  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 7, 4, 32)  │          0 │ max_pooling2d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 6, 3, 64)  │      8,256 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 3, 1, 64)  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 3, 1, 64)  │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 192)       │          0 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 192)       │     37,056 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 192)       │          0 │ flatten[0][0],    │
│                     │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 128)       │     24,704 │ multiply[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 64)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 5)         │        325 │ dropout_3[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 78,917 (308.27 KB)

 Trainable params: 78,917 (308.27 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate class weights
class_weights = compute_class_weight('balanced', classes=np.unique(y_train_smote), y=y_train_smote)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

# Train the model
history = cnn_attention_model.fit(X_train_reshaped, y_train_smote,
                                  epochs=20,
                                  batch_size=256,
                                  validation_data=(X_test_reshaped, y_test),
                                  class_weight=class_weight_dict,
                                  callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)])


Epoch 1/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 19s 14ms/step - accuracy: 0.5596 - loss: 1.0570 - val_accuracy: 0.7665 - val_loss: 0.6808
Epoch 2/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.8238 - loss: 0.4874 - val_accuracy: 0.8435 - val_loss: 0.4750
Epoch 3/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.8678 - loss: 0.3806 - val_accuracy: 0.8656 - val_loss: 0.3973
Epoch 4/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.8895 - loss: 0.3210 - val_accuracy: 0.8798 - val_loss: 0.3422
Epoch 5/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.9023 - loss: 0.2892 - val_accuracy: 0.9041 - val_loss: 0.2791
Epoch 6/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - accuracy: 0.9119 - loss: 0.2613 - val_accuracy: 0.8862 - val_loss: 0.3007
Epoch 7/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.9177 - loss: 0.2449 - val_accuracy: 0.8934 - val_loss: 0.2786
Epoch 8/20
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 15s 13ms/step - accuracy: 0.9228 -

In [5]:
from sklearn.metrics import classification_report

# Evaluate the model on the test set
test_loss, test_accuracy = cnn_attention_model.evaluate(X_test_reshaped, y_test)
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")

# Make predictions and generate a classification report
y_pred = np.argmax(cnn_attention_model.predict(X_test_reshaped), axis=-1)
classification_report_str = classification_report(y_test, y_pred, target_names=['N', 'S', 'V', 'F', 'Q'])
print("Classification Report:\n", classification_report_str)


548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9421 - loss: 0.1627
Final Test Accuracy: 94.23%
548/548 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Classification Report:
               precision    recall  f1-score   support

           N       0.99      0.94      0.97     14494
           S       0.43      0.89      0.58       445
           V       0.91      0.93      0.92      1158
           F       0.33      0.87      0.48       128
           Q       0.95      0.99      0.97      1286

    accuracy                           0.94     17511
   macro avg       0.72      0.92      0.78     17511
weighted avg       0.97      0.94      0.95     17511

